In [ ]:
import polars as pl
import numpy as np
import pandas as pd

In [2]:
import gc

gc.collect()

0

In [ ]:
data = pl.read_csv("matriz_binarizadaF.csv")
data.head(5)

In [6]:
df = data.to_pandas()
df.head(2)

,,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
0,200312-B21-A_GCAGCCATCGACCAAT-1_00482428,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,200312-B21-A_ACGGTTAAGTAAACTG-1_00482428,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
gc.collect()

17

In [7]:


# Descobrir o nome da primeira coluna (a das células) para podermos ignorá-la.
# (Em alguns arquivos ela não tem nome no cabeçalho, então pegamos dinamicamente)
coluna_celulas = data.columns[0]

# 2. Calcular a frequência
# - drop(): remove a coluna com os nomes das células
# - sum(): soma todas as colunas restantes (1 linha de resultado com a soma de cada gene)
somas = data.drop(coluna_celulas).sum()

# 3. Organizar os dados
# O resultado de sum() é horizontal (1 linha e milhares de colunas). 
# Usamos unpivot() (antigo melt) para transformar isso em 2 colunas visuais: "gene" e "frequencia"
df_frequencias = somas.unpivot(variable_name="gene", value_name="frequencia")

# 4. Ordenar e extrair os 5.000 maiores
top_5000_genes = (
    df_frequencias
    .sort("frequencia", descending=True) # Ordena do maior para o menor
    .head(5000)                          # Pega os 5000 primeiros
)

# Mostrar o resultado
print(top_5000_genes)

# 5. Salvar o resultado final em um novo arquivo (se quiser)
top_5000_genes.write_csv("top_5000_frequentes.csv")

shape: (5_000, 2)
┌─────────┬────────────┐
│ gene    ┆ frequencia │
│ ---     ┆ ---        │
│ str     ┆ i64        │
╞═════════╪════════════╡
│ MALAT1  ┆ 40782      │
│ CADM2   ┆ 39065      │
│ PCDH9   ┆ 39012      │
│ SNHG14  ┆ 38439      │
│ DLG2    ┆ 38107      │
│ …       ┆ …          │
│ METTL26 ┆ 11033      │
│ NDUFAF5 ┆ 11031      │
│ CALU    ┆ 11030      │
│ MED28   ┆ 11027      │
│ DYNLT3  ┆ 11027      │
└─────────┴────────────┘


In [18]:
lista_genes = pl.read_csv("top_5000_frequentes.csv")
lista_genes.head(5)

gene,frequencia
str,i64
"""MALAT1""",40782
"""CADM2""",39065
"""PCDH9""",39012
"""SNHG14""",38439
"""DLG2""",38107


In [21]:
minha_lista_genes = lista_genes["gene"].to_list()
print(minha_lista_genes[:10])  # Mostrar os 10 primeiros genes da lista

['MALAT1', 'CADM2', 'PCDH9', 'SNHG14', 'DLG2', 'MT-CO3', 'MT-ND3', 'ANK2', 'LSAMP', 'MAGI2']


In [23]:
import polars as pl

# 1. Ler o arquivo original (o que tem todas as colunas)


# 2. A sua lista com as colunas de interesse
# (Pode ser aquela lista de genes que você gerou, ou uma lista digitada à mão)
minha_lista_de_interesse = minha_lista_genes

# 3. PASSO DE SEGURANÇA: Garantir que só vamos pedir colunas que realmente existem no arquivo
# Isso evita o erro de pedir um gene que não está lá
colunas_validas = [data.columns[0]] + [col for col in minha_lista_de_interesse if col in data.columns]

# 4. O filtro mágico: selecionar apenas as colunas da lista
df_filtrado = data.select(colunas_validas)

# 5. Salvar em um novo arquivo
df_filtrado.write_csv("novo_arquivo_somente_interesse.csv")

# Uma mensagem para confirmar que deu tudo certo:
print(f"Pronto! O novo arquivo foi salvo com {len(df_filtrado.columns)} colunas.")

Pronto! O novo arquivo foi salvo com 5001 colunas.


In [24]:
df = pl.read_csv('novo_arquivo_somente_interesse.csv')
df.head(5)

,MALAT1,CADM2,PCDH9,SNHG14,DLG2,MT-CO3,MT-ND3,ANK2,LSAMP,MAGI2,ADGRB3,DST,MT-CO2,MT-ATP6,SNAP25,RTN4,CALM1,MT-CO1,FTX,MT-ND4,TCF4,MT-ND2,MT-ND1,PPP2R2B,CALM2,LRP1B,PTPRD,ANKS1B,NRXN1,CTNND2,SYT1,AUTS2,ADGRL3,CLU,CLASP2,GPM6A,…,MMP24OS,TNFRSF25,RUFY1,RLIM,PLEKHA2,SLC18B1,B3GALNT2,ZBTB11,TSC2,LINC00486,DHX57,KLHL4,EIF2S2,RBM28,ZFAND1,KDSR,SUDS3,NONO,NFXL1,PLPP3,MCC,ZFR2,FAM78B,TBC1D9B,SF3B3,CAMTA2,NDUFC1,CWC22,KCNN3,NT5DC3,DEF8,MRPL3,METTL26,NDUFAF5,CALU,MED28,DYNLT3
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""200312-B21-A_GCAGCCATCGACCAAT-…",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,…,0,1,1,1,0,1,1,1,1,0,0,0,0,1,1,0,1,0,1,0,1,1,0,0,0,0,0,1,0,1,0,1,1,0,1,0,1
"""200312-B21-A_ACGGTTAAGTAAACTG-…",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,…,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,1,0,0,1,1,1,1,0,1,1,0,0,1,1,1,0,1,1,1,0,0,0
"""200312-B21-A_CAATTTCAGATGCTTC-…",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,…,0,1,1,1,1,0,0,1,0,0,1,0,0,1,1,1,1,0,1,0,1,0,1,1,1,1,1,1,0,1,0,0,1,1,0,1,1
"""200312-B21-A_ACAAGCTAGCATTTCG-…",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,…,0,1,1,0,1,0,1,1,0,1,1,1,0,1,1,1,0,0,1,0,1,1,1,1,1,1,1,1,1,0,0,1,1,1,0,0,0
"""200312-B21-A_TACTTCAGTCATAGTC-…",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,…,1,0,1,1,0,0,1,0,0,1,1,1,1,0,1,0,1,0,1,1,0,1,1,1,1,0,0,0,0,1,0,1,0,1,1,0,0


In [25]:
df.shape

(40913, 5001)